# 01 · Synthetic project data generator

Writes the six project datasets the Blue Ridge LNG Terminal Phase 2 build ingests, as flat CSV files in the landing Volume `/Volumes/mfg_mc_se_sa/bluerridge_landing/project/`. Everything is synthetic (Faker + numpy) and deterministic: `random.seed(42)`, `np.random.seed(42)` and `Faker.seed(42)` are set at the top, every dataset cell re-seeds so it can be re-run on its own, and every date hangs off the `as_of_date` widget (default `2026-11-06`). The same inputs always produce identical files. Faker is the only dependency beyond the serverless defaults; the first cell pins its version so the text it generates cannot drift.

| File | Rows | Shape |
|---|---|---|
| `schedule.csv` | 500 | milestones with planned and actual dates, `depends_on` (`;`-separated predecessor ids) and a 1 to 5 `criticality` |
| `cost.csv` | 120 | cost line items: budget, committed, actual, estimate at completion (EAC), variance |
| `change_orders.csv` | 20 | change orders CO-101 to CO-120, each against one cost line |
| `ncrs.csv` | 80 | non-conformance reports, exactly 24 with `status = 'open'` |
| `safety_log.csv` | 400 | near misses, observations, first-aid, recordable and other events |
| `daily_reports.csv` | 180 | one free-text site report per working day |

The numbers are calibrated to the story in the Co-Pilot dashboard mockup (design spec section 5.4), so the KPI strip built later on this data reads: 6 of 11 key milestones hit (the cryogenic tank install is 4 days overdue), budget EAC +2.3% ($4.8M over plan, driven by materials-inflation change orders 112 to 118), 24 open NCRs (9 compliance, 15 workmanship) and 28 days since the last incident. Each dataset cell asserts its own numbers.

Besides the Volume, the notebook writes to `outputs/` next to itself: a five-row sample of each dataset (`sample_<name>.csv`, evenly spaced rows so they span the timeline) and `row_counts.json`.

In [1]:
%pip install -q faker==40.44.0

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
import json
import random
import sys
from datetime import date, timedelta
from pathlib import Path

import faker
import numpy as np
import pandas as pd
from faker import Faker

# Widgets let a job override these; the defaults match the variables in databricks.yml.
dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")
dbutils.widgets.text("as_of_date", "2026-11-06", "As-of date (YYYY-MM-DD)")
catalog = dbutils.widgets.get("catalog")
prefix = dbutils.widgets.get("schema_prefix")
AS_OF = date.fromisoformat(dbutils.widgets.get("as_of_date"))

# Determinism. Same seeds + same as-of date -> byte-identical CSVs.
SEED = 42
random.seed(42)
np.random.seed(42)
Faker.seed(42)
fake = Faker("en_US")


def reseed(salt):
    """Each dataset cell re-seeds, so it yields the same rows however often it is re-run."""
    random.seed(SEED + salt)
    np.random.seed(SEED + salt)
    fake.seed_instance(SEED + salt)


def pick(options, p=None):
    """Seeded (optionally weighted) choice that returns a plain Python value."""
    return options[int(np.random.choice(len(options), p=p))]


def person():
    return f"{fake.first_name()} {fake.last_name()}"


LANDING = f"{prefix}_landing"
VOLUME = f"/Volumes/{catalog}/{LANDING}/project"
OUT = Path("outputs")  # next to this notebook: sample CSVs and row_counts.json
PROJECT_START = AS_OF - timedelta(days=400)  # Phase 2 mobilised about 13 months before the as-of date

# Shared vocabulary. Disciplines match gold_pm.compliance_findings.discipline so the datasets join.
DISCIPLINES = ["piping", "structural", "mechanical", "electrical", "instr"]
AREAS = ["Train 3", "Train 4", "Tank T-203", "Tank T-204", "Jetty 2", "Pipe Rack North",
         "Pipe Rack South", "Utilities Block", "Main Substation", "Flare and Vent"]


def publish(name, pdf):
    """Write <name>.csv to the Volume as one flat file, save a 5-row sample next to the notebook, read the file back."""
    path = f"{VOLUME}/{name}.csv"
    pdf.to_csv(path, index=False)
    OUT.mkdir(exist_ok=True)
    pdf.iloc[np.linspace(0, len(pdf) - 1, 5).round().astype(int)].to_csv(OUT / f"sample_{name}.csv", index=False)
    return spark.read.option("header", True).option("inferSchema", True).csv(path)


print(f"catalog={catalog}  prefix={prefix}  as_of={AS_OF}  project_start={PROJECT_START}  volume={VOLUME}")
print(f"python {sys.version.split()[0]}  numpy {np.__version__}  pandas {pd.__version__}  faker {faker.VERSION}")

catalog=mfg_mc_se_sa  prefix=bluerridge  as_of=2026-11-06  project_start=2025-10-02  volume=/Volumes/mfg_mc_se_sa/bluerridge_landing/project
python 3.10.12  numpy 1.23.5  pandas 1.5.3  faker 40.44.0


## 1. Schema and Volume

Both statements are `IF NOT EXISTS` (the `00_setup` notebook already created them), so this notebook also runs on its own.

In [3]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{LANDING}` COMMENT 'Raw landing zone: Volumes holding the drawing PDFs and the project CSVs'")
print(f"schema  {catalog}.{LANDING}")

schema  mfg_mc_se_sa.bluerridge_landing


In [4]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{LANDING}`.`project` COMMENT 'Synthetic project data CSVs (schedule, cost, NCRs, change orders, safety log, daily reports)'")
print(f"volume  {VOLUME}")

volume  /Volumes/mfg_mc_se_sa/bluerridge_landing/project


## 2. Schedule: 500 milestones with dependencies

Eleven **key milestones** ("gates", `criticality = 5`) form the backbone. The other 489 work-package milestones each feed one gate, finish before that gate's review window, and depend on one or two earlier milestones (finish-to-start with a lag). `depends_on` holds `;`-separated `milestone_id`s and always points at a lower id, so the graph is acyclic. Actual dates are simulated forward through the graph, so a late predecessor pushes its successors; piping and mechanical work slips more as the project goes on, which gives the later `ai_forecast` step a trend to find.

`criticality` is an ordinal 1 to 5: gates are 5, and every other milestone is ranked by how many milestones sit downstream of it (the last work feeding a gate is at least 4).

As of the as-of date, gates 1 to 6 finished on time, gate 7 (the cryogenic tank install) is 4 days past its planned end and not done, and gates 8 to 11 are ahead.

In [5]:
reseed(1)

# The 11 key milestones. Offsets are days from AS_OF, so the story holds for any as-of date.
GATES = [
    ("Site Preparation and Piling Complete", "structural", -330),
    ("Foundations and Underground Utilities Complete", "structural", -265),
    ("Pipe Rack Steel Erection Complete", "structural", -195),
    ("Long-Lead Equipment Delivered to Site", "mechanical", -145),
    ("Liquefaction Train 3 Module Setting Complete", "mechanical", -85),
    ("Main Substation Energized", "electrical", -35),
    ("Cryogenic Tank T-203 Install Complete", "mechanical", -4),  # 4 days overdue: the slip in the Daily Brief
    ("Cryogenic Piping Welding and NDE Complete", "piping", 70),
    ("Instrumentation Loop Checks Complete", "instr", 140),
    ("Mechanical Completion Certified", "mechanical", 220),
    ("Commissioning Start (Cooldown Ready)", "piping", 300),
]
# Activities per discipline, ordered from early to late in the build; a package only draws from its own phase.
ACTIVITIES = {
    "structural": ["Drive piles", "Install rebar", "Pour foundation", "Backfill and compact", "Erect steel", "Grout baseplates", "Fireproof steel"],
    "piping": ["Fabricate spools", "Install supports", "Erect piping", "Weld and inspect", "Hydrotest line", "Insulate cryogenic line"],
    "mechanical": ["Expedite long-lead equipment", "Set equipment", "Erect tank shell course", "Install heat exchanger", "Install compressor skid", "Align rotating equipment", "Grout equipment"],
    "electrical": ["Install grounding grid", "Install underground duct bank", "Install cable tray", "Set transformer", "Pull power cable", "Terminate cables", "Energize MCC"],
    "instr": ["Review control system design", "Install control cabinets", "Install field instruments", "Pull instrument cable", "Calibrate transmitters", "Loop check", "Function test safety system"],
}
BASE_SLIP = {"piping": 5, "mechanical": 4, "instr": 3, "structural": 1.5, "electrical": 1}  # typical days late
N_ROWS, LAG, OVERDUE_GATE = 500, 10, 6


def day(n):
    return timedelta(days=int(n))


def activity(disc, g):
    """A work activity from the discipline's list, drawn from the window that matches gate g's phase of the build."""
    acts = ACTIVITIES[disc]
    centre = g / (len(GATES) - 1) * (len(acts) - 1)
    return pick([a for k, a in enumerate(acts) if abs(k - centre) <= 1.5])


# 1. Milestones: 11 gates plus 489 work packages spread over the gate windows.
gate_end = [AS_OF + day(off) for _, _, off in GATES]
ms = [dict(pkg=g, gate=True, name=name, discipline=disc, ps=gate_end[g] - day(14), pe=gate_end[g])
      for g, (name, disc, _) in enumerate(GATES)]
lo = [max(PROJECT_START + day(15), (gate_end[g - 1] if g else PROJECT_START) - day(20)) for g in range(len(GATES))]
span = [(gate_end[g] - day(24) - lo[g]).days for g in range(len(GATES))]
per_gate = np.random.multinomial(N_ROWS - len(GATES), np.array(span) / sum(span))
wp = 0
for g, k in enumerate(per_gate):
    w = np.full(len(DISCIPLINES), 0.5 / 4)
    w[DISCIPLINES.index(GATES[g][1])] = 0.5  # half of each package is the gate's own discipline
    for _ in range(k):
        wp += 1
        disc = pick(DISCIPLINES, w)
        pe = lo[g] + day(np.random.randint(0, span[g] + 1))
        ps = max(PROJECT_START, pe - day(np.clip(np.random.gamma(3.0, 7.0), 5, 60)))
        ms.append(dict(pkg=g, gate=False, name=f"{activity(disc, g)} - {pick(AREAS)} (WP-{wp:03d})",
                       discipline=disc, ps=ps, pe=pe))

# 2. Ids follow planned start, so every predecessor (finishes LAG days before its successor starts) has a lower id.
ms.sort(key=lambda m: (m["ps"], m["pe"], not m["gate"], m["name"]))
n_ms = len(ms)
ids = [f"MS-{i + 1:03d}" for i in range(n_ms)]
gate_at = {m["pkg"]: i for i, m in enumerate(ms) if m["gate"]}

# 3. Dependencies.
preds = [[] for _ in range(n_ms)]
for i, m in enumerate(ms):
    ok = [j for j in range(i) if ms[j]["pe"] + day(LAG) <= m["ps"]]
    if m["gate"]:  # a gate needs the previous gate plus the two last-finishing work packages feeding it
        feeders = sorted((j for j in ok if ms[j]["pkg"] == m["pkg"] and not ms[j]["gate"]), key=lambda j: ms[j]["pe"])[-2:]
        preds[i] = sorted(([gate_at[m["pkg"] - 1]] if m["pkg"] else []) + feeders)
    else:  # 0 to 2 of the six most recently finished milestones in this or the previous package
        near = sorted((j for j in ok if ms[j]["pkg"] in (m["pkg"], m["pkg"] - 1)), key=lambda j: ms[j]["pe"])[-6:]
        k = min(len(near), pick([0, 1, 2], [0.25, 0.55, 0.20]))
        if k:
            preds[i] = sorted(int(j) for j in np.random.choice(near, size=k, replace=False))

# 4. Criticality: gates 5, the rest ranked by transitive downstream count (the work that feeds a gate is >= 4).
succ = [[] for _ in range(n_ms)]
for i, p in enumerate(preds):
    for j in p:
        succ[j].append(i)
down = [set() for _ in range(n_ms)]
for i in reversed(range(n_ms)):  # successors have higher ids, so a reverse sweep has already seen them
    for s in succ[i]:
        down[i] |= {s} | down[s]
cuts = np.percentile([len(down[i]) for i in range(n_ms) if not ms[i]["gate"]], [40, 70, 90])
crit = [5 if m["gate"] else 1 + int(sum(len(down[i]) > c for c in cuts)) for i, m in enumerate(ms)]
for g_i in gate_at.values():
    for j in preds[g_i]:
        if not ms[j]["gate"]:
            crit[j] = max(crit[j], 4)

# 5. Actual dates, simulated forward in id order and then cut off at AS_OF (the future is unknown).
a_start, a_end = [None] * n_ms, [None] * n_ms
for i, m in enumerate(ms):
    early = m["pkg"] < OVERDUE_GATE
    if m["gate"]:
        if early:  # gates 1-6 finish on time, up to 3 days early
            a_start[i], a_end[i] = m["ps"] + day(np.random.randint(-2, 3)), m["pe"] - day(np.random.randint(0, 4))
        else:  # the tank gate runs about a week late; later gates are still in the future
            a_start[i], a_end[i] = m["ps"] + day(np.random.randint(0, 3)), m["pe"] + day(np.random.randint(5, 13))
        continue
    t = (m["pe"] - PROJECT_START).days / 730  # share of the 2-year horizon; piping and mechanical slip more as it goes on
    trend = (2.5 if early else 10) * t if m["discipline"] in ("piping", "mechanical") else 0
    mu = 0.3 * BASE_SLIP[m["discipline"]] - 0.5 + trend if early else 1.6 * BASE_SLIP[m["discipline"]] + trend
    lo_s, hi_s = (-4, 5) if early else (-3, 50)
    s_end = int(np.clip(round(float(np.random.normal(mu, 2.5 if early else 7.0))), lo_s, hi_s))
    s_start = int(round(s_end * float(np.random.uniform(0.3, 0.9))))
    start = max(PROJECT_START, m["ps"] + day(s_start))
    if preds[i]:  # a late predecessor pushes this milestone's start
        start = max(start, max(a_end[j] for j in preds[i]) + day(1))
    a_start[i], a_end[i] = start, start + day(max(2, (m["pe"] - m["ps"]).days + s_end - s_start))

schedule = pd.DataFrame({
    "milestone_id": ids,
    "name": [m["name"] for m in ms],
    "discipline": [m["discipline"] for m in ms],
    "planned_start": [m["ps"] for m in ms],
    "planned_end": [m["pe"] for m in ms],
    "actual_start": [a if a <= AS_OF else None for a in a_start],
    "actual_end": [a if a <= AS_OF else None for a in a_end],
    "depends_on": [";".join(ids[j] for j in p) for p in preds],
    "criticality": crit,
})

# Checks: the numbers the KPI strip will show.
ps_, pe_ = pd.to_datetime(schedule.planned_start), pd.to_datetime(schedule.planned_end)
as_, ae_ = pd.to_datetime(schedule.actual_start), pd.to_datetime(schedule.actual_end)
key = schedule.criticality == 5
hit = int((key & (ae_ <= pe_)).sum())
tank = schedule[schedule.name.str.startswith("Cryogenic Tank")].iloc[0]
assert len(schedule) == N_ROWS and schedule.milestone_id.is_unique and int(key.sum()) == 11
assert hit == 6 and pd.isna(tank.actual_end) and (AS_OF - tank.planned_end).days == 4
assert (ps_ <= pe_).all() and ((as_ <= ae_) | ae_.isna()).all() and (as_.notna() | ae_.isna()).all()
assert all(int(d[3:]) < int(i[3:]) for i, dep in zip(schedule.milestone_id, schedule.depends_on) if dep for d in dep.split(";"))
for i, p in enumerate(preds):  # a finished milestone started after everything it depends on finished
    assert ae_.iloc[i] is pd.NaT or all(ae_.iloc[j] < as_.iloc[i] for j in p)

df = publish("schedule", schedule)
print(f"schedule: {len(schedule)} rows, {len(df.columns)} columns -> {VOLUME}/schedule.csv")
print(f"  key milestones hit on time: {hit} of {int(key.sum())}; "
      f"complete {int(ae_.notna().sum())}, in progress {int((as_.notna() & ae_.isna()).sum())}, not started {int(as_.isna().sum())}")
print(f"  late finishes among complete milestones: {int((ae_ > pe_).sum())}; criticality counts: {schedule.criticality.value_counts().sort_index().to_dict()}")
display(df.limit(5))
print("The 11 key milestones (criticality 5):")
display(df.filter("criticality = 5").orderBy("planned_end"))

schedule: 500 rows, 9 columns -> /Volumes/mfg_mc_se_sa/bluerridge_landing/project/schedule.csv
  key milestones hit on time: 6 of 11; complete 275, in progress 18, not started 207
  late finishes among complete milestones: 146; criticality counts: {1: 220, 2: 127, 3: 80, 4: 62, 5: 11}


milestone_id,name,discipline,planned_start,planned_end,actual_start,actual_end,depends_on,criticality
MS-001,Install underground duct bank - Jetty 2 (WP-001),electrical,2025-10-02,2025-10-19,2025-10-04,2025-10-22,,1
MS-002,Drive piles - Tank T-204 (WP-005),structural,2025-10-02,2025-10-20,2025-10-02,2025-10-20,,4
MS-003,Install rebar - Jetty 2 (WP-010),structural,2025-10-02,2025-10-21,2025-10-02,2025-10-21,,2
MS-004,Install underground duct bank - Main Substation (WP-019),electrical,2025-10-02,2025-10-23,2025-10-05,2025-10-27,,4
MS-005,Review control system design - Tank T-203 (WP-011),instr,2025-10-05,2025-10-21,2025-10-04,2025-10-20,,1


The 11 key milestones (criticality 5):


milestone_id,name,discipline,planned_start,planned_end,actual_start,actual_end,depends_on,criticality
MS-042,Site Preparation and Piling Complete,structural,2025-11-27,2025-12-11,2025-11-28,2025-12-08,MS-021;MS-028,5
MS-082,Foundations and Underground Utilities Complete,structural,2026-01-31,2026-02-14,2026-02-01,2026-02-11,MS-042;MS-058;MS-069,5
MS-141,Pipe Rack Steel Erection Complete,structural,2026-04-11,2026-04-25,2026-04-12,2026-04-23,MS-082;MS-098;MS-127,5
MS-182,Long-Lead Equipment Delivered to Site,mechanical,2026-05-31,2026-06-14,2026-05-31,2026-06-13,MS-141;MS-152;MS-168,5
MS-231,Liquefaction Train 3 Module Setting Complete,mechanical,2026-07-30,2026-08-13,2026-07-31,2026-08-12,MS-182;MS-204;MS-209,5
MS-268,Main Substation Energized,electrical,2026-09-18,2026-10-02,2026-09-17,2026-09-29,MS-231;MS-249;MS-252,5
MS-281,Cryogenic Tank T-203 Install Complete,mechanical,2026-10-19,2026-11-02,2026-10-21,,MS-262;MS-268;MS-270,5
MS-340,Cryogenic Piping Welding and NDE Complete,piping,2027-01-01,2027-01-15,,,MS-281;MS-306;MS-310,5
MS-396,Instrumentation Loop Checks Complete,instr,2027-03-12,2027-03-26,,,MS-340;MS-377;MS-382,5
MS-457,Mechanical Completion Certified,mechanical,2027-05-31,2027-06-14,,,MS-396;MS-411;MS-430,5


## 3. Cost: 120 line items

Cost line items per discipline, category and area, with the budget, what is committed, what is spent and the estimate at completion (`eac_usd`). `variance_usd = eac_usd - budget_usd` (positive means over plan). Two totals are fixed by design: the budget sums to $208.7M and the variance to $4.8M, which is +2.3%. About two thirds of the overrun sits on the seven biggest material lines (materials inflation); the rest is small plus-and-minus noise on the other lines. The 20 change orders in the next section hang off these lines.

In [6]:
reseed(2)

BUDGET_TOTAL, OVERRUN_TOTAL, N_INFLATED = 208_700_000, 4_800_000, 7  # +2.3% EAC, $4.8M over plan
SIZE = {"material": 1.6, "equipment": 2.2, "subcontract": 2.0, "labor": 1.0, "engineering": 0.6, "indirect": 0.5}
CODES = {"piping": "PIP", "structural": "STR", "mechanical": "MEC", "electrical": "ELE", "instr": "INS",
         "material": "MAT", "equipment": "EQP", "subcontract": "SUB", "labor": "LAB", "engineering": "ENG", "indirect": "IND"}
SCOPE = {"piping": "Cryogenic piping", "structural": "Structural steel and concrete", "mechanical": "Rotating and static equipment",
         "electrical": "Power and distribution", "instr": "Instrumentation and controls"}
WHAT = {"material": "material supply", "equipment": "equipment and rentals", "subcontract": "subcontract package",
        "labor": "installation labor", "engineering": "engineering support", "indirect": "field indirects"}
SELF = {"labor": "EPC self-perform", "engineering": "EPC engineering", "indirect": "EPC site services"}


def tidy_order(line):
    return DISCIPLINES.index(line[0]), list(SIZE).index(line[1]), AREAS.index(line[2])


def supplier():
    return f"{fake.last_name()} {pick(['Industries', 'Fabrication', 'Supply', 'Controls', 'Cryogenics', 'Constructors'])}"


# 120 distinct (discipline, category, area) combinations out of the 300 possible.
lines = sorted(random.sample([(d, c, a) for d in DISCIPLINES for c in SIZE for a in AREAS], 120), key=tidy_order)
disciplines, categories, areas = (list(col) for col in zip(*lines))

# Budget: lognormal sizes scaled to the fixed total, in whole thousands of dollars.
weight = np.array([SIZE[c] for c in categories]) * np.random.lognormal(0, 0.8, len(lines))
budget = (np.round(weight / weight.sum() * BUDGET_TOTAL / 1000) * 1000).astype(np.int64)
budget[budget.argmax()] += BUDGET_TOTAL - budget.sum()

# Variance: the biggest material lines take the inflation (65% of the overrun); every other line gets small noise
# plus a budget-weighted share of whatever is needed to land exactly on the total.
inflated = [int(i) for i in np.argsort(-budget) if categories[i] == "material"][:N_INFLATED]
rest = [i for i in range(len(lines)) if i not in inflated]
var = np.zeros(len(lines))
bump = np.random.uniform(0.05, 0.09, N_INFLATED) * budget[inflated]
var[inflated] = bump * (0.65 * OVERRUN_TOTAL / bump.sum())
noise = np.random.normal(0, 0.02, len(rest)) * budget[rest]
var[rest] = noise + (OVERRUN_TOTAL - var[inflated].sum() - noise.sum()) * budget[rest] / budget[rest].sum()
variance = (np.round(var / 1000) * 1000).astype(np.int64)
variance[rest[int(budget[rest].argmax())]] += OVERRUN_TOTAL - variance.sum()

eac = budget + variance
committed = (np.round(eac * np.random.uniform(0.70, 1.0, len(lines)) / 1000) * 1000).astype(np.int64)
actual = (np.round(committed * np.random.beta(2.2, 2.2, len(lines)) / 1000) * 1000).astype(np.int64)

cost = pd.DataFrame({
    "cost_id": [f"CST-{i + 1:03d}" for i in range(len(lines))],
    "cost_code": [f"{CODES[d]}-{CODES[c]}-{i + 1:03d}" for i, (d, c, _) in enumerate(lines)],
    "description": [f"{SCOPE[d]} {WHAT[c]} - {a}" for d, c, a in lines],
    "discipline": disciplines,
    "category": categories,
    "area": areas,
    "vendor": [SELF.get(c) or supplier() for c in categories],
    "budget_usd": budget,
    "committed_usd": committed,
    "actual_usd": actual,
    "eac_usd": eac,
    "variance_usd": variance,
    "report_date": AS_OF,
})

# Checks: the KPI strip numbers and the money identities.
assert len(cost) == 120 and cost.cost_id.is_unique and cost.cost_code.is_unique and cost.description.is_unique
assert cost.budget_usd.sum() == BUDGET_TOTAL and cost.variance_usd.sum() == OVERRUN_TOTAL
assert (cost.variance_usd == cost.eac_usd - cost.budget_usd).all()
assert (cost.actual_usd <= cost.committed_usd).all() and (cost.committed_usd <= cost.eac_usd).all()

df = publish("cost", cost)
print(f"cost: {len(cost)} rows, {len(df.columns)} columns -> {VOLUME}/cost.csv")
print(f"  budget ${cost.budget_usd.sum() / 1e6:,.1f}M, EAC ${cost.eac_usd.sum() / 1e6:,.1f}M: "
      f"+{cost.variance_usd.sum() / cost.budget_usd.sum():.1%}, ${cost.variance_usd.sum() / 1e6:.1f}M over plan; "
      f"spent ${cost.actual_usd.sum() / 1e6:,.1f}M, committed ${cost.committed_usd.sum() / 1e6:,.1f}M")
print(f"  {int((cost.variance_usd > 0).sum())} lines over plan, {int((cost.variance_usd < 0).sum())} under")
display(df.limit(5))

cost: 120 rows, 13 columns -> /Volumes/mfg_mc_se_sa/bluerridge_landing/project/cost.csv
  budget $208.7M, EAC $213.5M: +2.3%, $4.8M over plan; spent $84.1M, committed $183.3M
  74 lines over plan, 43 under


cost_id,cost_code,description,discipline,category,area,vendor,budget_usd,committed_usd,actual_usd,eac_usd,variance_usd,report_date
CST-001,PIP-MAT-001,Cryogenic piping material supply - Train 4,piping,material,Train 4,Ortiz Cryogenics,909000,932000,436000,940000,31000,2026-11-06
CST-002,PIP-MAT-002,Cryogenic piping material supply - Tank T-203,piping,material,Tank T-203,Weaver Fabrication,4749000,5069000,2174000,5138000,389000,2026-11-06
CST-003,PIP-MAT-003,Cryogenic piping material supply - Tank T-204,piping,material,Tank T-204,Atkins Industries,4489000,4771000,2081000,4890000,401000,2026-11-06
CST-004,PIP-MAT-004,Cryogenic piping material supply - Jetty 2,piping,material,Jetty 2,Thompson Cryogenics,459000,464000,202000,483000,24000,2026-11-06
CST-005,PIP-MAT-005,Cryogenic piping material supply - Flare and Vent,piping,material,Flare and Vent,Perez Fabrication,512000,519000,227000,546000,34000,2026-11-06


## 4. Change orders: 20 rows

CO-101 to CO-120, each against one cost line, numbered in the order they were requested. CO-112 to CO-118 are the **materials-inflation** orders from the Daily Brief story: all approved, one per inflated material line, each worth exactly that line's variance. The other approved orders are sized to part of the variance on a fitting line (so approved change orders on a line never exceed its overrun); CO-109, 110, 119 and 120 are still pending and CO-104 and 107 were rejected, so they add nothing to the EAC.

In [7]:
reseed(3)

INFLATION = {"piping": "Cryogenic pipe and fittings price escalation", "structural": "Structural steel price escalation",
             "mechanical": "Equipment material surcharge", "electrical": "Copper cable price escalation",
             "instr": "Instrument tubing and cable price escalation"}
OTHER = {  # reason -> (titles, cost categories it usually lands on, schedule impact range in days)
    "design_change": (["Revised piping routing after P&ID revision", "Relocated equipment skid per updated plot plan"], ["engineering", "subcontract"], (3, 21)),
    "scope_addition": (["Additional fireproofing at pipe rack", "Added spare pump and isolation valves"], ["subcontract", "equipment"], (0, 14)),
    "site_conditions": (["Unforeseen rock excavation at foundation", "Dewatering for high groundwater"], ["subcontract", "labor"], (5, 30)),
    "schedule_acceleration": (["Overtime to recover cryogenic piping slip", "Second shift for tank shell welding"], ["labor"], (-10, 0)),
    "regulatory": (["Seismic anchorage update after code revision", "Added fire and gas detection coverage"], ["engineering", "equipment"], (0, 10)),
}
PENDING, REJECTED, INFLATION_COS = {109, 110, 119, 120}, {104, 107}, range(112, 119)
requested = sorted(PROJECT_START + timedelta(days=int(d)) for d in np.random.randint(45, 386, 20))
room = dict(zip(cost.cost_id, cost.variance_usd))  # overrun not yet explained by an approved change order
by_cost_id = cost.set_index("cost_id")
requesters = [person() for _ in range(6)]
inflation_ids = [cost.cost_id.iloc[i] for i in inflated]  # the 7 inflated material lines, biggest budget first

rows = []
for k, num in enumerate(range(101, 121)):
    status = "pending" if num in PENDING else "rejected" if num in REJECTED else "approved"
    if num in INFLATION_COS:
        cost_id = inflation_ids[num - 112]
        reason, amount, impact = "materials_inflation", room[cost_id], 0
        title = f"{INFLATION[by_cost_id.discipline[cost_id]]} - {by_cost_id.area[cost_id]}"
    else:
        reason = list(OTHER)[num % len(OTHER)]
        titles, cats, (lo_d, hi_d) = OTHER[reason]
        fit = [c for c in cost.cost_id if by_cost_id.category[c] in cats and c not in inflation_ids
               and (status != "approved" or room[c] >= 50_000)]
        cost_id = pick(fit)
        amount = int(round((room[cost_id] * np.random.uniform(0.3, 0.7) if status == "approved" else np.random.uniform(40_000, 600_000)) / 1000) * 1000)
        impact, title = int(np.random.randint(lo_d, hi_d + 1)), f"{pick(titles)} - {by_cost_id.area[cost_id]}"
    if status == "approved":
        room[cost_id] -= amount
    decided = None if status == "pending" else min(requested[k] + timedelta(days=int(np.random.randint(7, 36))), AS_OF - timedelta(days=1))
    rows.append((f"CO-{num}", cost_id, title, reason, status, amount, impact, requested[k], decided, pick(requesters)))
change_orders = pd.DataFrame(rows, columns=["co_id", "cost_id", "title", "reason", "status", "amount_usd",
                                            "schedule_impact_days", "requested_date", "decided_date", "requested_by"])

# Checks: the Daily Brief story and the link back to the cost lines.
approved = change_orders[change_orders.status == "approved"]
explained = approved.groupby("cost_id").amount_usd.sum()
infl = change_orders[change_orders.reason == "materials_inflation"]
assert len(change_orders) == 20 and change_orders.co_id.is_unique and set(change_orders.cost_id) <= set(cost.cost_id)
assert list(infl.co_id) == [f"CO-{n}" for n in INFLATION_COS] and (infl.status == "approved").all()
assert (explained <= by_cost_id.variance_usd.reindex(explained.index)).all() and (change_orders.amount_usd > 0).all()
assert (pd.to_datetime(change_orders.decided_date).fillna(pd.Timestamp(AS_OF)) <= pd.Timestamp(AS_OF)).all()

df = publish("change_orders", change_orders)
print(f"change_orders: {len(change_orders)} rows, {len(df.columns)} columns -> {VOLUME}/change_orders.csv")
print(f"  status: {change_orders.status.value_counts().to_dict()}")
print(f"  CO-112 to CO-118 (materials inflation): ${infl.amount_usd.sum() / 1e6:.2f}M = {infl.amount_usd.sum() / OVERRUN_TOTAL:.0%} of the ${OVERRUN_TOTAL / 1e6:.1f}M overrun; "
      f"all approved change orders: ${approved.amount_usd.sum() / 1e6:.2f}M")
display(df.limit(5))

change_orders: 20 rows, 10 columns -> /Volumes/mfg_mc_se_sa/bluerridge_landing/project/change_orders.csv
  status: {'approved': 14, 'pending': 4, 'rejected': 2}
  CO-112 to CO-118 (materials inflation): $3.12M = 65% of the $4.8M overrun; all approved change orders: $3.48M


co_id,cost_id,title,reason,status,amount_usd,schedule_impact_days,requested_date,decided_date,requested_by
CO-101,CST-053,Added spare pump and isolation valves - Tank T-203,scope_addition,approved,28000,4,2025-12-01,2025-12-30,Larry Paul
CO-102,CST-014,Dewatering for high groundwater - Main Substation,site_conditions,approved,20000,15,2025-12-03,2025-12-18,Joseph Freeman
CO-103,CST-062,Overtime to recover cryogenic piping slip - Utilities Block,schedule_acceleration,approved,48000,0,2025-12-10,2026-01-02,Larry Paul
CO-104,CST-083,Added fire and gas detection coverage - Jetty 2,regulatory,rejected,266000,9,2025-12-18,2025-12-27,Justin Wilson
CO-105,CST-012,Revised piping routing after P&ID revision - Tank T-203,design_change,approved,20000,21,2026-01-12,2026-01-28,Charles Smith


## 5. NCRs: 80 non-conformance reports, exactly 24 open

56 closed and **24 open**. The open ones follow the dashboard story: 9 are compliance NCRs (they cite the same codes the drawing audit plants flaws against) and 15 are workmanship NCRs, concentrated in piping, structural and instrumentation. Each NCR points at a work-package milestone of its own discipline: one that was live when the NCR was raised, or else the one that finished closest to that date.

In [8]:
reseed(4)

FLAWS = {  # compliance NCRs mirror the flaw types planted in the drawing set: (title, code citation, disciplines)
    "flange": ("Flange rating mismatch on piping isometric", "ASME B16.5 §4.2", ("piping", "mechanical")),
    "loto": ("Missing lockout/tagout notation on equipment drawing", "OSHA 1910.147", ("mechanical", "electrical")),
    "egress": ("Egress width below code minimum on platform layout", "IBC 1005.1", ("structural",)),
    "weld": ("Weld joint specification mismatch on line list", "ASME B31.3 §341", ("piping",)),
    "relief": ("Pressure relief notation missing on P&ID", "ASME B31.3 §322", ("piping", "instr")),
}
WORKMANSHIP = {
    "piping": ["Weld rejected on radiography for cryogenic spool", "Pipe support misaligned beyond tolerance", "Hydrotest leak at flanged joint"],
    "structural": ["Honeycombing found at foundation pour", "Anchor bolt projection out of tolerance", "Steel bolting torque below requirement"],
    "mechanical": ["Rotating equipment alignment out of tolerance", "Tank shell weld distortion beyond limit", "Voids found in baseplate grout"],
    "electrical": ["Cable tray support spacing non-conforming", "Termination torque not recorded", "Grounding conductor undersized as installed"],
    "instr": ["Transmitter installed at wrong elevation", "Instrument tubing not supported to standard", "Loop check failed: wiring polarity reversed"],
}
OTHER_TITLES = {
    "materials": ["Material test report missing heat number", "Wrong material grade delivered against purchase order", "Coating damage on delivered material"],
    "documentation": ["As-built markup not submitted", "Inspection record incomplete", "Weld map not updated after rework"],
}
SEVERITY_DAYS = {"critical": 14, "major": 30, "minor": 60}  # days allowed to close

# 24 open (9 compliance + 15 workmanship) and 56 closed.
open_cases = [("compliance", True)] * 9 + [("workmanship", True)] * 15
closed_cases = [("compliance", False)] * 12 + [("workmanship", False)] * 24 + [("materials", False)] * 12 + [("documentation", False)] * 8
random.shuffle(closed_cases)
disc_open = ["piping"] * 9 + ["structural"] * 6 + ["instr"] * 4 + ["mechanical"] * 3 + ["electrical"] * 2  # pipe, structural, instrument
random.shuffle(disc_open)
disc_closed = np.random.choice(DISCIPLINES, len(closed_cases), p=[0.30, 0.22, 0.20, 0.15, 0.13]).tolist()
inspectors = [person() for _ in range(6)]
leads = {d: person() for d in DISCIPLINES}
work_packages = schedule[schedule.criticality < 5]

rows = []
for (cat, is_open), disc in zip(open_cases + closed_cases, disc_open + disc_closed):
    if cat == "compliance":
        title, ref, _ = FLAWS[pick([k for k, v in FLAWS.items() if disc in v[2]])]
    else:
        title, ref = pick(WORKMANSHIP[disc] if cat == "workmanship" else OTHER_TITLES[cat]), ""
    raised = (AS_OF - timedelta(days=int(np.random.randint(1, 46)))) if is_open else (PROJECT_START + timedelta(days=int(np.random.randint(30, 376))))
    severity = pick(list(SEVERITY_DAYS), [0.10, 0.40, 0.50])
    closed = None if is_open else min(raised + timedelta(days=int(np.random.randint(4, 51))), AS_OF - timedelta(days=1))
    # Link to a work-package milestone of this discipline that was live when the NCR was raised.
    mine = work_packages[work_packages.discipline == disc]
    live = mine[(mine.planned_start <= raised) & (mine.planned_end >= raised - timedelta(days=30))]
    if len(live):
        milestone = live.milestone_id.iloc[int(np.random.randint(len(live)))]
    else:  # no work of this discipline was live: take the one that finished closest to the raised date
        milestone = mine.milestone_id.iloc[int(np.argmin([abs((e - raised).days) for e in mine.planned_end]))]
    area, found_by = pick(AREAS), pick(inspectors)
    rows.append(dict(
        title=title,
        description=f"{title} at {area}, found by {found_by}. " + (f"Cited requirement: {ref}. " if ref else "") + ("Awaiting disposition." if is_open else "Dispositioned and closed after re-inspection."),
        discipline=disc, category=cat, severity=severity, status="open" if is_open else "closed", area=area,
        raised_date=raised, due_date=raised + timedelta(days=SEVERITY_DAYS[severity]), closed_date=closed,
        raised_by=found_by, assigned_to=leads[disc], milestone_id=milestone, code_reference=ref))
ncrs = pd.DataFrame(rows).sort_values(["raised_date", "title"], kind="mergesort").reset_index(drop=True)
ncrs.insert(0, "ncr_id", [f"NCR-{i + 1:04d}" for i in range(len(ncrs))])

# Checks: the open-NCR tile and the foreign key.
open_ = ncrs[ncrs.status == "open"]
assert len(ncrs) == 80 and ncrs.ncr_id.is_unique and len(open_) == 24
assert open_.category.value_counts().to_dict() == {"workmanship": 15, "compliance": 9}
assert open_.closed_date.isna().all() and ncrs[ncrs.status == "closed"].closed_date.notna().all()
assert set(ncrs.milestone_id) <= set(schedule.milestone_id)

df = publish("ncrs", ncrs)
print(f"ncrs: {len(ncrs)} rows, {len(df.columns)} columns -> {VOLUME}/ncrs.csv")
print(f"  open: {len(open_)} ({open_.category.value_counts().to_dict()}); by discipline: {open_.discipline.value_counts().to_dict()}")
print(f"  open NCRs past their due date: {int((pd.to_datetime(open_.due_date) < pd.Timestamp(AS_OF)).sum())}")
display(df.limit(5))

ncrs: 80 rows, 15 columns -> /Volumes/mfg_mc_se_sa/bluerridge_landing/project/ncrs.csv
  open: 24 ({'workmanship': 15, 'compliance': 9}); by discipline: {'piping': 9, 'structural': 6, 'instr': 4, 'mechanical': 3, 'electrical': 2}
  open NCRs past their due date: 9


ncr_id,title,description,discipline,category,severity,status,area,raised_date,due_date,closed_date,raised_by,assigned_to,milestone_id,code_reference
NCR-0001,Anchor bolt projection out of tolerance,"Anchor bolt projection out of tolerance at Tank T-204, found by Brandy Valentine. Dispositioned and closed after re-inspection.",structural,workmanship,major,closed,Tank T-204,2025-11-12,2025-12-12,2025-12-18,Brandy Valentine,Samantha Rose,MS-006,
NCR-0002,Instrument tubing not supported to standard,"Instrument tubing not supported to standard at Pipe Rack North, found by Jeremy Black. Dispositioned and closed after re-inspection.",instr,workmanship,major,closed,Pipe Rack North,2025-11-12,2025-12-12,2025-11-17,Jeremy Black,Timothy Moore,MS-018,
NCR-0003,Inspection record incomplete,"Inspection record incomplete at Utilities Block, found by Luke Bryan. Dispositioned and closed after re-inspection.",piping,documentation,major,closed,Utilities Block,2025-11-15,2025-12-15,2025-12-30,Luke Bryan,Megan Taylor,MS-013,
NCR-0004,As-built markup not submitted,"As-built markup not submitted at Train 4, found by Pamela Ross. Dispositioned and closed after re-inspection.",piping,documentation,critical,closed,Train 4,2025-11-20,2025-12-04,2025-11-28,Pamela Ross,Megan Taylor,MS-013,
NCR-0005,Missing lockout/tagout notation on equipment drawing,"Missing lockout/tagout notation on equipment drawing at Pipe Rack North, found by Antonio Jones. Cited requirement: OSHA 1910.147. Dispositioned and closed after re-inspection.",electrical,compliance,minor,closed,Pipe Rack North,2025-11-23,2026-01-22,2025-12-26,Antonio Jones,Christopher Mccann,MS-024,OSHA 1910.147


## 6. Safety log: 400 events

Leading indicators (near misses, unsafe conditions and acts, stop-work calls) and lagging ones (first-aid, recordable and lost-time injuries, property damage, environmental releases). `is_incident` is true for the lagging ones. The last incident is exactly **28 days before the as-of date**, which is the "days without incident" figure; leading-indicator events keep arriving after it. Activity ramps up through the build, so there are more events in recent months.

In [9]:
reseed(5)

EVENTS = {  # type -> (weight, is_incident, severity, details)
    "near_miss": (0.29, False, None, ["unsecured scaffold plank shifted underfoot", "suspended load swung toward crew", "tool dropped from elevated platform",
                                      "vehicle reversed without a spotter", "worker nearly stepped into an unbarricaded trench"]),
    "unsafe_condition": (0.26, False, "low", ["missing guardrail on laydown platform", "damaged lifting sling found in use", "uncontrolled cable across walkway",
                                              "blocked emergency eyewash station", "unlabelled chemical container"]),
    "unsafe_act": (0.14, False, "low", ["fall protection not tied off above 6 ft", "worker in the line of fire near a crane pick", "hot work without a fire watch",
                                        "equipment guard bypassed", "welding without required PPE"]),
    "stop_work": (0.10, False, "medium", ["pick stopped after wind exceeded limit", "hot work halted: permit not posted", "excavation stopped: shoring not inspected",
                                          "lift halted: tag lines not rigged", "confined space entry halted: gas test expired"]),
    "first_aid": (0.10, True, "medium", ["minor laceration to hand", "dust irritation flushed at eyewash", "sprained wrist after slip",
                                         "heat stress symptoms treated on site", "bruised shin from falling material"]),
    "property_damage": (0.05, True, "medium", ["forklift struck a pipe rack support", "crane boom contacted scaffold", "light vehicle backed into conduit rack"]),
    "environmental": (0.02, True, "medium", ["hydraulic oil release contained within berm", "diesel spill during refuelling", "sediment-laden runoff from stockpile"]),
    "recordable": (0.03, True, "high", ["fractured finger in pinch point", "laceration needing sutures", "back strain with restricted duty", "eye injury needing medical treatment"]),
    "lost_time": (0.01, True, "critical", ["fall from ladder with sprained ankle", "crush injury to hand", "shoulder strain from manual handling"]),
}
ROOT_CAUSES = ["inadequate planning", "housekeeping", "procedure not followed", "equipment failure", "inadequate supervision", "training gap", "weather", "communication"]
ACTIONS = ["toolbox talk held", "barricade installed", "crew retrained", "job hazard analysis updated", "equipment replaced", "supervision increased", "procedure revised"]
N_EVENTS, DAYS_CLEAR = 400, 28
reporters = [person() for _ in range(8)]

kinds = list(EVENTS)
types = np.random.choice(kinds, N_EVENTS, p=[EVENTS[k][0] for k in kinds]).tolist()
is_incident = np.array([EVENTS[t][1] for t in types])
# More events in recent months (triangular ramp); incidents stop DAYS_CLEAR days before the as-of date.
horizon = (AS_OF - PROJECT_START).days - 14
offset = np.round(np.random.triangular(0, 0.7, 1.0, N_EVENTS) * np.where(is_incident, horizon - DAYS_CLEAR, horizon)).astype(int)
dates = [PROJECT_START + timedelta(days=14 + int(o)) for o in offset]
last = max((i for i in range(N_EVENTS) if is_incident[i]), key=lambda i: dates[i])
dates[last] = AS_OF - timedelta(days=DAYS_CLEAR)  # pin the most recent incident to exactly 28 days ago

rows = []
for d, t in sorted(zip(dates, types), key=lambda x: x[0]):
    _, incident, severity, details = EVENTS[t]
    area = pick(AREAS)
    rows.append(dict(
        event_date=d, event_type=t, is_incident=incident,
        severity=severity or pick(["low", "medium", "high"], [0.55, 0.35, 0.10]),
        area=area, discipline=pick(DISCIPLINES, [0.28, 0.22, 0.22, 0.16, 0.12]),
        description=f"{t.replace('_', ' ').capitalize()}: {pick(details)} ({area})",
        root_cause=pick(ROOT_CAUSES), corrective_action=pick(ACTIONS), reported_by=pick(reporters),
        days_lost=int(np.random.randint(1, 22)) if t == "lost_time" else 0))
safety_log = pd.DataFrame(rows)
safety_log.insert(0, "log_id", [f"SAF-{i + 1:04d}" for i in range(len(safety_log))])

# Checks: the safety tile.
incidents = safety_log[safety_log.is_incident]
days_clear = (AS_OF - max(incidents.event_date)).days
assert len(safety_log) == N_EVENTS and safety_log.log_id.is_unique and days_clear == DAYS_CLEAR
assert safety_log.event_date.max() <= AS_OF and (safety_log[~safety_log.is_incident].event_date.max() > AS_OF - timedelta(days=DAYS_CLEAR))
assert (safety_log.days_lost > 0).sum() == (safety_log.event_type == "lost_time").sum()

df = publish("safety_log", safety_log)
print(f"safety_log: {len(safety_log)} rows, {len(df.columns)} columns -> {VOLUME}/safety_log.csv")
print(f"  {len(incidents)} incidents, {N_EVENTS - len(incidents)} leading-indicator events; days since last incident: {days_clear}")
print(f"  by type: {safety_log.event_type.value_counts().to_dict()}")
display(df.limit(5))

safety_log: 400 rows, 12 columns -> /Volumes/mfg_mc_se_sa/bluerridge_landing/project/safety_log.csv
  99 incidents, 301 leading-indicator events; days since last incident: 28
  by type: {'near_miss': 112, 'unsafe_condition': 101, 'unsafe_act': 52, 'first_aid': 44, 'stop_work': 36, 'property_damage': 21, 'recordable': 15, 'environmental': 15, 'lost_time': 4}


log_id,event_date,event_type,is_incident,severity,area,discipline,description,root_cause,corrective_action,reported_by,days_lost
SAF-0001,2025-11-05,property_damage,True,medium,Tank T-203,structural,Property damage: crane boom contacted scaffold (Tank T-203),equipment failure,job hazard analysis updated,Joseph Sheppard,0
SAF-0002,2025-11-07,unsafe_act,False,low,Pipe Rack South,piping,Unsafe act: hot work without a fire watch (Pipe Rack South),procedure not followed,equipment replaced,Joseph Sheppard,0
SAF-0003,2025-12-01,near_miss,False,low,Train 3,piping,Near miss: unsecured scaffold plank shifted underfoot (Train 3),equipment failure,job hazard analysis updated,Shawn Cooper,0
SAF-0004,2025-12-02,near_miss,False,low,Train 3,structural,Near miss: worker nearly stepped into an unbarricaded trench (Train 3),weather,procedure revised,William Long,0
SAF-0005,2025-12-03,first_aid,True,medium,Pipe Rack South,piping,First aid: sprained wrist after slip (Pipe Rack South),inadequate planning,barricade installed,Michael Jordan,0


## 7. Daily reports: 180 free-text site reports

One report for each of the last 180 working days up to the as-of date. `work_summary` names the milestones that were actually in progress (or finished) that day in the schedule above, `delays_and_issues` ties to real NCRs and suppliers, and `field_notes` is free text from `Faker.text()` drawn from a site-vocabulary word list, standing in for the superintendent's unstructured remarks that the daily-brief narrator reads.

In [10]:
reseed(6)

SITE_WORDS = ["weld", "flange", "spool", "hydrotest", "torque", "crane", "scaffold", "rigging", "grout", "rebar", "formwork", "cable", "tray", "conduit",
              "termination", "transmitter", "calibration", "loop", "insulation", "cryogenic", "tank", "shell", "nozzle", "support", "inspection", "radiography",
              "punchlist", "permit", "toolbox", "laydown", "delivery", "module", "alignment", "coating", "backfill", "survey", "shift", "crew", "foreman",
              "gasket", "valve", "bolt", "plate", "steel", "concrete", "pour", "cure", "test", "lift", "pipe", "pump", "trench", "pile", "safety"]
WEATHER = ["Clear", "Partly cloudy", "Overcast", "Light rain", "Heavy rain", "High wind", "Fog", "Hot and humid"]
WEATHER_P = [0.30, 0.22, 0.18, 0.12, 0.05, 0.06, 0.04, 0.03]
WEATHER_ISSUE = {"Heavy rain": "Outdoor work suspended for rain; half a shift lost on exposed scopes.",
                 "High wind": "Crane picks held while wind stayed above lift limits.",
                 "Fog": "Late start on tank and jetty scopes until visibility cleared."}
ITEMS = ["cryogenic valves", "pipe spools", "transformer bushings", "instrument cable", "structural steel", "heat exchanger tubes", "tank roof plates"]

work = list(zip(schedule.name, schedule.actual_start, schedule.actual_end, schedule.criticality < 5))  # (name, start, end, is_work_package)
suppliers = sorted({v for v in cost.vendor if not v.startswith("EPC")})
superintendents = [person() for _ in range(5)]

rows = []
for ts in pd.bdate_range(end=AS_OF, periods=180):  # the last 180 working days
    d = ts.date()
    finished = [n for n, s, e, wp_ in work if e == d][:3]
    active = [n for n, s, e, wp_ in work if wp_ and s is not None and s <= d and (e is None or e > d)]
    crews = random.sample(active, min(3, len(active)))
    summary = ("Crews advanced " + "; ".join(crews) + "." if crews else "Limited field activity.") + (" Completed today: " + "; ".join(finished) + "." if finished else "")
    weather = pick(WEATHER, WEATHER_P)
    recent = ncrs[(ncrs.raised_date <= d) & (ncrs.raised_date >= d - timedelta(days=30))].ncr_id.tolist()
    options = ["Waiting on RFI response from engineering", "Crane downtime for repairs", f"Late delivery of {pick(ITEMS)} from {pick(suppliers)}",
               f"Permit to work delayed at {pick(AREAS)}"]
    options += [f"Rework underway on {pick(recent)}", f"Inspection hold point not released on {pick(recent)}"] if recent else []
    issue = WEATHER_ISSUE.get(weather) or (pick(options) + "." if np.random.rand() < 0.5 else "")
    notes = " ".join(fake.text(max_nb_chars=int(np.random.randint(120, 320)), ext_word_list=SITE_WORDS).split())  # one line, no embedded newlines
    rows.append(dict(
        report_date=d, author=pick(superintendents), weather=weather,
        temp_f=int(round(58 + 22 * np.sin(2 * np.pi * (d.timetuple().tm_yday - 105) / 365) + np.random.normal(0, 5))),
        headcount=int(650 + 700 * (d - PROJECT_START).days / 400 + np.random.normal(0, 35)),
        work_summary=summary, delays_and_issues=issue, field_notes=notes))
daily_reports = pd.DataFrame(rows)
daily_reports.insert(0, "report_id", [f"DR-{i + 1:04d}" for i in range(len(daily_reports))])

# Checks: one report per working day, ending on the as-of date (or the Friday before it), no embedded newlines.
dates_ = pd.to_datetime(daily_reports.report_date)
assert len(daily_reports) == 180 and daily_reports.report_id.is_unique and dates_.is_unique and (dates_.dt.weekday < 5).all()
assert dates_.max().date() <= AS_OF and not daily_reports.select_dtypes("object").apply(lambda c: c.astype(str).str.contains("[\\n\\r\"]")).any().any()

df = publish("daily_reports", daily_reports)
print(f"daily_reports: {len(daily_reports)} rows, {len(df.columns)} columns -> {VOLUME}/daily_reports.csv")
print(f"  {daily_reports.report_date.min()} to {daily_reports.report_date.max()}; {int((daily_reports.delays_and_issues != '').sum())} reports flag a delay or issue")
display(df.limit(5))

daily_reports: 180 rows, 9 columns -> /Volumes/mfg_mc_se_sa/bluerridge_landing/project/daily_reports.csv
  2026-03-02 to 2026-11-06; 109 reports flag a delay or issue


report_id,report_date,author,weather,temp_f,headcount,work_summary,delays_and_issues,field_notes
DR-0001,2026-03-02,Amy Anderson,Clear,42,952,Crews advanced Set equipment - Main Substation (WP-108); Install rebar - Tank T-204 (WP-077); Drive piles - Jetty 2 (WP-099).,,Survey cable permit pour grout cable hydrotest. Rigging shift trench hydrotest. Pump crew crane tray termination. Pump delivery transmitter. Cryogenic support formwork plate.
DR-0002,2026-03-03,Shawna Lee,Overcast,35,950,Crews advanced Drive piles - Train 3 (WP-092); Install rebar - Jetty 2 (WP-113); Review control system design - Train 3 (WP-119). Completed today: Drive piles - Train 4 (WP-110); Install field instruments - Train 4 (WP-120); Drive piles - Main Substation (WP-073).,Late delivery of cryogenic valves from Ortiz Cryogenics.,Grout grout termination concrete module gasket support. Laydown pipe module bolt foreman weld inspection.
DR-0003,2026-03-04,Tammy Russo,Partly cloudy,45,923,Crews advanced Expedite long-lead equipment - Tank T-204 (WP-067); Install field instruments - Train 4 (WP-066); Review control system design - Jetty 2 (WP-074). Completed today: Review control system design - Pipe Rack North (WP-078); Install rebar - Tank T-204 (WP-077); Install rebar - Pipe Rack North (WP-114).,,Pipe torque grout safety hydrotest test shift. Cure cryogenic shell laydown flange.
DR-0004,2026-03-05,Kathleen Martin,Clear,41,917,Crews advanced Install control cabinets - Train 4 (WP-097); Erect piping - Tank T-204 (WP-115); Erect tank shell course - Utilities Block (WP-098). Completed today: Install field instruments - Train 4 (WP-066).,,Hydrotest module torque. Weld transmitter coating concrete scaffold concrete crane. Laydown permit module loop shell calibration.
DR-0005,2026-03-06,Tammy Russo,Fog,49,919,Crews advanced Install rebar - Jetty 2 (WP-113); Pour foundation - Tank T-203 (WP-068); Drive piles - Flare and Vent (WP-075). Completed today: Install rebar - Tank T-203 (WP-091); Erect tank shell course - Utilities Block (WP-098); Erect piping - Tank T-204 (WP-115).,Late start on tank and jetty scopes until visibility cleared.,Cable shift laydown toolbox. Formwork pipe spool support rigging punchlist trench. Tray hydrotest bolt. Tank spool punchlist gasket lift.


## 8. Verify the Volume and write `row_counts.json`

Every CSV is read back from the Volume (not counted from memory), so `row_counts.json` records what actually landed. The last checks are the foreign keys between datasets.

In [11]:
datasets = {"schedule": schedule, "cost": cost, "ncrs": ncrs, "change_orders": change_orders, "safety_log": safety_log, "daily_reports": daily_reports}

print(f"{VOLUME}/")
for f in dbutils.fs.ls(VOLUME):
    print(f"  {f.name:<20} {f.size:>9,} bytes")

counts = {name: spark.read.option("header", True).csv(f"{VOLUME}/{name}.csv").count() for name in datasets}
assert counts == {name: len(pdf) for name, pdf in datasets.items()}, counts

# Foreign keys and the numbers the KPI strip will show.
assert set(change_orders.cost_id) <= set(cost.cost_id) and set(ncrs.milestone_id) <= set(schedule.milestone_id)
assert all(d in set(schedule.milestone_id) for dep in schedule.depends_on if dep for d in dep.split(";"))
assert counts["ncrs"] == 80 and int((ncrs.status == "open").sum()) == 24
print(f"open NCRs: {int((ncrs.status == 'open').sum())}; key milestones hit: 6 of 11; EAC +{cost.variance_usd.sum() / cost.budget_usd.sum():.1%}; "
      f"days since last incident: {(AS_OF - max(safety_log[safety_log.is_incident].event_date)).days}")

OUT.mkdir(exist_ok=True)
(OUT / "row_counts.json").write_text(json.dumps(counts, indent=2) + "\n")
print(json.dumps(counts, indent=2))

/Volumes/mfg_mc_se_sa/bluerridge_landing/project/
  change_orders.csv        2,790 bytes
  cost.csv                20,955 bytes
  daily_reports.csv       77,637 bytes
  ncrs.csv                23,757 bytes
  safety_log.csv          72,727 bytes
  schedule.csv            51,500 bytes
open NCRs: 24; key milestones hit: 6 of 11; EAC +2.3%; days since last incident: 28
{
  "schedule": 500,
  "cost": 120,
  "ncrs": 80,
  "change_orders": 20,
  "safety_log": 400,
  "daily_reports": 180
}
